In [ ]:
# BLOCO 1: Instalações que Exigem Reinício

# 1. Downgrade preventivo do Numpy (para evitar conflitos com AV)
!pip install "numpy<2.0"

# 2. Instala AV (biblioteca de decodificação) na versão estável (evita o Silent Hang do MP3)
!pip install "av==15.0.0"

print("\n-------------------------------------------------------------------------")
print("✅ INSTALAÇÕES INICIAIS CONCLUÍDAS.")
print("🔴 AÇÃO CRÍTICA: Você DEVE REINICIAR o ambiente agora para estabilizar o numpy.")
print("Vá em Runtime (Ambiente de Execução) > Restart session (Reiniciar Sessão) ou clique no botão que aparecer.")
print("-------------------------------------------------------------------------")



In [ ]:
# BLOCO 2: Instalações Finais (Execute SOMENTE APÓS o REINÍCIO da Sessão do Bloco 1)

# 1. Força a reinstalação do PyTorch + TorchVision sincronizados
#    A versão 2.3.1 é compatível com o ambiente CUDA 11.8 da GPU T4 do Colab.
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

# 2. Instala WhisperX e dependências manualmente 
!pip install git+https://github.com/m-bain/whisperX.git --no-deps
!pip install ffmpeg-python pandas scipy "pyannote.audio==3.3.2" ctranslate2 faster-whisper

print("\n✅ INSTALAÇÃO E CONFIGURAÇÃO COMPLETAS. O ambiente está pronto.")

In [ ]:
# BLOCO 3: MONTAGEM DO DRIVE E CONFIGURAÇÃO DE SESSÃO
from google.colab import drive
drive.mount('/content/drive')

import os

# ----------------------------------------------------------------------
# EDITE APENAS ESTAS DUAS VARIÁVEIS PARA CADA NOVA SESSÃO:
# ----------------------------------------------------------------------

# O ID da sessão atual (e.g., "THoL01", "ID56", "THoL03")
ID_SESSAO = "THoL03" 

# O caminho base da sua pasta de projeto no Google Drive
PASTA_PROJETO = "/content/drive/MyDrive/rpg-transcription"

# ----------------------------------------------------------------------
# VARIÁVEIS DERIVADAS (Não precisam ser editadas)
# ----------------------------------------------------------------------

# Arquivo de entrada para a transcrição
ARQUIVO_WAV_ENTRADA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}w.wav")

# Arquivo TXT de saída da transcrição (será lido pelo Gemini)
ARQUIVO_TXT_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_transcricao_final.txt")

# Arquivo Markdown (Crônica) de saída do Gemini
ARQUIVO_CRONICA_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_cronica.md")


print("Configuração de sessão concluída:")
print(f"  ID da Sessão: {ID_SESSAO}")
print(f"  Entrada WAV: {ARQUIVO_WAV_ENTRADA}")
print(f"  Saída TXT: {ARQUIVO_TXT_SAIDA}")
print(f"  Saída Crônica: {ARQUIVO_CRONICA_SAIDA}")

# ----------------------------------------------------------------------
# VARIÁVEIS DE CONTEÚDO (Podem ser editadas conforme mais conteúdo precisar ser adicionado)
# ----------------------------------------------------------------------

# Glossário para ajudar o Whisper a entender nomes próprios e termos das sessões
NOMES_DA_CAMPANHA = [
# Termos de sistema
"D20",
"Iniciativa",
"Perception",
"Insight",
"Investigation",
# Nomes de personagens de jogadores - Zéfiros
"Pharah",
"Janos",
"Bjorn",
"Bbo",
"Akros",
# Nomes de personagens de jogadores - Inferno
"Kay",
"Sarfan Thuranni",
"Vander Bremen",
"Purcival Purcy Bartolomeow",
"Penélope",
"Orog",
# Nomes de outros personagens
"De Lata",
"Pedrinha",
"Alustriel",
"Mordenkainen",
"Tasha",
"Zéfiros",
"Tiamat",
"Zariel",
"Vecna",
# Nomes de locais
"Eberron",
"Elturel",
"Elturgard",
"Baldur’s Gate",
"Candlekeep",
"Icewind Dale",
"Baróvia",
"Faerûn"
]

GLOSSARIO_NOMES = ", ".join(NOMES_DA_CAMPANHA)


In [ ]:
# BLOCO 4: Execução da Transcrição (Faster-Whisper)
from faster_whisper import WhisperModel
print("Iniciando a transcrição com Faster-Whisper.")

# 1a. Carregar o modelo (usa a GPU e o motor ctranslate2)
model = WhisperModel("large-v2", device="cuda", compute_type="float16")

# 1b. Carregar o modelo (usa a CPU e o motor ctranslate2)
# model = WhisperModel("large-v2", device="cpu", compute_type="int8")

# 2. Transcrição
segments, info = model.transcribe(
    ARQUIVO_WAV_ENTRADA, # Usa a variável global
    language="pt", 
    vad_filter=True, 
    word_timestamps=False, 
    initial_prompt=GLOSSARIO_NOMES 
)

# 3. Processar e salvar o texto
# O caminho completo de saída já foi definido em ARQUIVO_TXT_SAIDA no Bloco 1
with open(ARQUIVO_TXT_SAIDA, "w", encoding="utf-8") as f:
    print(f"Salvando o texto em: {ARQUIVO_TXT_SAIDA}")
    
    # Escreve cada segmento como uma linha no arquivo
    for segment in segments:
        f.write(f"{segment.text}\n")
        
print("\nTranscrição CONCLUÍDA e salva no Google Drive!")



In [ ]:
# BLOCO 5: Geração da Crônica com Gemini API
import os
from google import genai
from google.genai.errors import APIError
from google.colab import userdata # Importa o módulo para acessar Secrets

# --- CHAVE DE API: NOVO MÉTODO DE ACESSO ---
# 1. Tenta acessar o Secret. A variável GEMINI_API_KEY será definida como o valor
#    do Secret, ou será None se o acesso falhar ou a chave não existir.

# O NOME DO SECRET DEVE SER EXATAMENTE 'GEMINI_API_KEY'
GEMINI_API_KEY = None # Inicializa como None por segurança

try:
    # Acessa o valor do Secret do Colab
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except KeyError:
    # Se o Secret não for encontrado, a variável permanece None e a mensagem é impressa.
    print("ERRO DE CONFIGURAÇÃO: O Secret 'GEMINI_API_KEY' não foi encontrado. Verifique se o nome está correto e se o acesso está ativado.")
# ------------------------------------------------------------

# As variáveis de caminho já estão definidas no Bloco 0
CAMINHO_TRANSCRICAO = ARQUIVO_TXT_SAIDA
CAMINHO_SAIDA_CRONICA = ARQUIVO_CRONICA_SAIDA

def gerar_cronica_gemini_automatizada():
    """Lê o arquivo de transcrição, gera a crônica e salva o resultado."""
   
    # Configura o cliente da API
    try:
        # Se a chave não for válida, o cliente pode falhar aqui
        client = genai.Client(api_key=GEMINI_API_KEY)
    except NameError:
        print("ERRO: A variável GEMINI_API_KEY não está definida. Cole sua chave de API antes de prosseguir.")
        return
    except Exception as e:
        print(f"ERRO: Falha ao configurar o cliente. Verifique se a chave de API é válida. Detalhes: {e}")
        return

    # 1. Ler o conteúdo do arquivo TXT do Drive
    try:
        with open(CAMINHO_TRANSCRICAO, 'r', encoding='utf-8') as f:
            transcricao_texto = f.read()
        print(f"Transcrição lida com sucesso. Tamanho: {len(transcricao_texto)} caracteres.")
    except FileNotFoundError:
        print(f"ERRO: Arquivo de transcrição não encontrado em {CAMINHO_TRANSCRICAO}. Verifique o caminho.")
        return

    # 2. Montar o Prompt com as instruções de Crônica

    referencia_lore = "\n".join([f"- {nome}" for nome in NOMES_DA_CAMPANHA])

    prompt_completo = f"""

    **BLOCO DE TEXTO TRANSCRITO:**
    --- INÍCIO DA TRANSCRIÇÃO ---
    {transcricao_texto}
    --- FIM DA TRANSCRIÇÃO —

    **Instrução Principal:**
    Atue como um Escriba e Cronista Mestre de RPG.

    Sua tarefa é ler a transcrição bruta da sessão de RPG fornecida acima e transformá-la em um resumo narrativo coeso.


    **IMPORTANTE:** Use o glossário a seguir para corrigir nomes que o transcritor possa ter entendido errado: {referencia_lore}

    **DIRETRIZES DE FLUXO E COBERTURA:**

    1. **Foco Narrativo:** O texto é uma transcrição de áudio, portanto pode conter erros de fala, repetições, gírias e conversas "off-game" (fora do personagem). Você deve filtrar esse ruído e extrair a essência da história jogada. Ignore mecânicas de dados e conversas off-game. Transforme os acontecimentos em uma crônica épica.
    2. **Varredura Completa:** Você deve processar o texto do início ao fim. Garanta que o último parágrafo do seu resumo corresponda aos eventos finais da história registrados na transcrição.
    3. **Sem Alucinações:** Se a sessão terminou abruptamente, encerre o texto onde a história parou. Não invente ganchos ou conclusões que não aconteceram.
    4. **Destaques:** Use **negrito** para nomes de personagens, lugares, itens importantes e momentos críticos.
    5. **Linguagem**: Use uma linguagem narrativa, clara e empolgante (como o resumo de um capítulo de livro).

    Instruções de Formatação:

    1. O resumo deve ter exatamente entre 2 e 3 parágrafos.
    2. O primeiro parágrafo deve cobrir o primeiro terço da sessão.
    3. O segundo parágrafo deve cobrir o segundo terço da sessão.
    4. O terceiro parágrafo deve cobrir o terceiro terço da sessão.
    5. Use Markdown para formatar o resultado (títulos com #, subtítulos com ##, negritos com **).

    Critérios de Conteúdo (O que incluir):
    1. Pontos Principais: Onde a história avançou? Qual era o objetivo da sessão?
    2. Decisões Chave: Escolhas difíceis ou caminhos que os jogadores decidiram seguir.
    3. Ações de Destaque: Mencione o que deu muito certo (sucessos críticos/jogadas inteligentes) e o que deu muito errado (falhas críticas/planos que falharam).
    4. Momentos Memoráveis: Se houver cenas engraçadas ou diálogos icônicos que marcaram a sessão, inclua-os brevemente para dar "sabor" ao resumo.

    O que IGNORAR:
    1. Discussões longas sobre regras ou matemática de dados (apenas o resultado narrativo importa).
    2. Conversas paralelas que não agregam à história ou ao humor da mesa.

    Após o resumo da sessão, adicione os seguintes pontos:
   ## NPCs encontrados: descreva brevemente os NPCs encontrados, por exemplo, “Kalyth, guerreira orc, e Tasha, maga poderosa”.
   ## Itens obtidos: o grupo obteve algum item importante na sessão?
   ## Momentos inspiradores: “Inspiração” é um termo usado no jogo para quando um personagem faz algo memorável, engraçado ou inteligente. Nesses casos, o mestre pode “dar uma inspiração para o personagem”. Caso um ou mais personagens tenham recebido inspiração durante a sessão, descreva neste item o que eles fizeram para receber essa inspiração.
   ## Quantos dias se passaram: normalmente ao final de um dia o grupo faz um descanso longo (long rest), enquanto em outras vezes só é mencionado que um novo dia está amanhecendo. Analise quantos dias se passaram nesta sessão, se algum. Se não for possível determinar com clareza, indique apenas que o tempo transcorrido é indeterminado.
   ## Onde a sessão parou: onde o grupo estava e o que estava fazendo nos minutos finais da sessão?

    """

    # 3. Chamar a API do Gemini
    print("\nEnviando solicitação ao modelo Gemini 2.5 Flash (Isso pode levar alguns segundos)...")
    generation_config = {
        "temperature": 0.7,
        "top_p": 0.95,
        "max_output_tokens": 8192,
    }

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt_completo,
            config=generation_config
        )
       
        cronica_final = response.text
       
        # 4. Verificação de Bloqueio (Safety Filter)
        if cronica_final is None:
            feedback = response.prompt_feedback
            if feedback and feedback.block_reason:
                print(f"\n❌ ERRO DE GERAÇÃO: O modelo não conseguiu gerar texto.")
                print(f"Motivo do Bloqueio (Safety Filter): {feedback.block_reason.name}")
            else:
                 print("\n⚠️ ERRO DE GERAÇÃO: O modelo não conseguiu gerar texto. (Motivo desconhecido ou timeout)")
            return

        # 5. Salvar a Crônica final no Drive
        with open(CAMINHO_SAIDA_CRONICA, 'w', encoding='utf-8') as f:
            f.write(cronica_final)
           
        print(f"\n✅ SUCESSO! Crônica gerada e salva no Drive em: {CAMINHO_SAIDA_CRONICA}")
       
    except APIError as e:
        print(f"ERRO DE API: Falha na chamada ao Gemini. Detalhes: {e}")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")

# Executar a função principal
gerar_cronica_gemini_automatizada()

